In [1]:
import requests
import pandas as pd
import time

In [ ]:
key = ""

In [12]:
base_url = "https://catalog.api.2gis.com/3.0/items"
headers = {"User-Agent": "Mozilla/5.0"}
location = "37.6173,55.7558"
radius = 100000

results = []

page = 1
while True:
    print(f"\nСтраница {page}")
    params = {
        "q": "пансионат для пожилых",
        "location": location,
        "radius": radius,
        "page": page,
        "page_size": 10,
        "fields": ",".join([
            "items.point",
            "items.address",
            "items.reviews",
            "items.schedule",
            "items.rubrics",
            "items.external_content"
        ]),

        "key": key
    }


    response = requests.get(
        base_url,
        params=params,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    items = data.get("result", {}).get("items", [])
    if not items:
        break
    for item in items:
        try:
            name = item.get("name")
            address = (item.get("full_address_name") or item.get("address_name"))
            rating = item.get("reviews", {}).get("rating")
            reviews = (item.get("reviews", {}).get("general_review_count"))

            point = item.get("point", {})
            lat = point.get("lat")
            lon = point.get("lon")

            rubrics = []
            for rubric in item.get("rubrics", []):
                rubric_name = rubric.get("name")
                if rubric_name:
                    rubrics.append(rubric_name)
            
            if "Дома престарелых" not in rubrics:
                continue

            results.append({
                "name": name,
                "address": address,
                "rating": rating,
                "reviews": reviews,
                "rubrics": " | ".join(rubrics),
                "lat": lat,
                "lon": lon
            })

        except Exception as e:
            print("Ошибка:", e)

    page += 1
    time.sleep(1)


df = pd.DataFrame(results)
df = df.drop_duplicates(subset=["name", "address"])
df = df.sort_values(by=["reviews", "rating"], ascending=False)

df.to_csv("2gis_pansionaty.csv", index=False, encoding="utf-8-sig")
print(f"\nВсего объектов: {len(df)}")


Страница 1

Страница 2

Страница 3

Страница 4

Страница 5

Страница 6

Всего объектов: 46
